# Colab Runner

This notebook only **runs** the project's scripts on Google Colab. All the logic lives in `src/camtrap/`, so the same pipeline runs on any computer.

Colab is used for the two heavy steps:

1. **Downloading the photos** (about 5 GB). Colab sits on Google's network, next to where the photos are stored.
2. **Training the models**, on Colab's free GPU (a processor built for the many small calculations a neural network needs).

**Before running:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

The photos are saved to Google Drive (`MyDrive/camtrap/data.zip`) the first time, so later sessions only unpack them.

### Step 1: Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Step 2: Get the project from GitHub

A fresh copy every session, so Colab always runs the latest committed code.

In [ ]:
REPO = "https://github.com/DianaAdhiambo1809/camtrap-species-classifier.git"

%cd /content
!rm -rf camtrap-species-classifier
!git clone -q {REPO}
%cd /content/camtrap-species-classifier
!pip install -q -e .
!git log --oneline -1

### Step 3: Run the data pipeline (first session) or unpack it from Drive (later sessions)

The four steps are the same commands listed in the README. One zip file on Drive is much faster to save and load than 8,000 separate photos.

In [ ]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/camtrap")
DRIVE.mkdir(parents=True, exist_ok=True)
saved = DRIVE / "data.zip"

if saved.exists():
    print("Unpacking data saved in an earlier session...")
    !unzip -q -o {saved} -d /content/camtrap-species-classifier
else:
    !python -m camtrap.data.download
    !python -m camtrap.data.clean
    !python -m camtrap.data.split
    !python -m camtrap.data.download_images
    print("Saving the processed tables and photos to Drive...")
    !zip -q -r {saved} data/processed data/images

print("Done")

### Step 4: Check the photos

In [ ]:
import pandas as pd

photos = list(Path("data/images").rglob("*.jpg"))
print(f"Photos on disk: {len(photos):,}")

# data/images/<split>/<class>/<photo>.jpg
table = pd.crosstab(pd.Series([p.parts[3] for p in photos], name="class"),
                    pd.Series([p.parts[2] for p in photos], name="split"))
print(table[["train", "val", "test"]])

failed = Path("data/processed/failed_downloads.csv")
if failed.exists():
    print(f"\nPhotos that could not be downloaded: {len(pd.read_csv(failed))}")

### Step 5: Image features for the baseline

A pretrained EfficientNetB0 turns each photo into 1,280 numbers. The result, `features.npz` (a few tens of MB), is copied to Drive. Download it from `MyDrive/camtrap/` into the local `data/processed/` folder to run the baselines on your own computer.

In [ ]:
if (DRIVE / "features.npz").exists():
    print("Features already on Drive; copying them back")
    !cp {DRIVE}/features.npz data/processed/
else:
    !python -m camtrap.models.features
    !cp data/processed/features.npz {DRIVE}/

!ls -lh data/processed/features.npz

### Step 6: Train the CNNs with Optuna

Each model takes roughly 30–45 minutes on a T4 GPU. Everything the training produces (models, Optuna results, charts, MLflow runs) is zipped to `MyDrive/camtrap/outputs.zip`. Download it and unzip it into the project folder on your computer.

In [ ]:
!pip install -q optuna mlflow

# Quick check (about 2 minutes) that the training script runs end to end, before the long runs
!python -m camtrap.models.train --model scratch --quick
!python -m camtrap.models.train --model pretrained --quick

In [ ]:
!python -m camtrap.models.train --model scratch

In [ ]:
!python -m camtrap.models.train --model pretrained

In [ ]:
# Predictions on the validation and test photos, for the evaluation on your computer
!python -m camtrap.models.predict

# Save everything the training produced to Drive in one file
!zip -q -r {DRIVE}/outputs.zip models reports mlflow.db
!ls -lh {DRIVE}/outputs.zip